In [ ]:
# In[1]:

import pandas as pd

# Load CSVs into DataFrames and parse timestamps to UTC datetimes
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

for df in (metric_df, trace_df, log_df, error_logs_df):
    # ensure timestamp column exists and convert
    if 'timestamp' in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to build summary and top-n name counts for each dataset
def summarize_df(df, name_col=None, sample_col=None):
    total_rows = len(df)
    distinct_cmdb = df['cmdb_id'].unique() if 'cmdb_id' in df.columns else []
    distinct_cmdb_count = len(distinct_cmdb)
    # limit cmdb list to 20 items for compactness
    cmdb_sample = list(distinct_cmdb[:20])
    min_ts = df['ts'].min() if 'ts' in df.columns else pd.NaT
    max_ts = df['ts'].max() if 'ts' in df.columns else pd.NaT
    top_names_df = pd.DataFrame()
    if name_col and name_col in df.columns:
        top_names_df = df[name_col].value_counts().head(20).reset_index()
        top_names_df.columns = [name_col, 'count']
    elif sample_col and sample_col in df.columns:
        top_names_df = df[sample_col].value_counts().head(20).reset_index()
        top_names_df.columns = [sample_col, 'count']
    summary = pd.DataFrame([{
        'file_rows': total_rows,
        'distinct_cmdb_count': distinct_cmdb_count,
        'cmdb_ids_sample': cmdb_sample,
        'min_ts': min_ts,
        'max_ts': max_ts
    }])
    return summary, top_names_df

# Metric file: use kpi_name
metric_summary, metric_top_kpis = summarize_df(metric_df, name_col='kpi_name')

# Trace file: use trace_name
trace_summary, trace_top_traces = summarize_df(trace_df, name_col='trace_name')

# Log file: use log_name
log_summary, log_top_logs = summarize_df(log_df, name_col='log_name')

# Error logs: show top message samples
err_summary, err_top_messages = summarize_df(error_logs_df, sample_col='message')

# Display compact aggregates only (up to top 20 rows for name counts)
metric_summary, metric_top_kpis, trace_summary, trace_top_traces, log_summary, log_top_logs, err_summary, err_top_messages

```
Out[1]:
```
summary = (
    "Summary of loaded telemetry files (UTC):\n\n"
    "metric.csv:\n"
    "- total rows: 8575\n"
    "- distinct cmdb_id count: 68 (sample: ts-admin-basic-info-service, ts-admin-order-service, ts-admin-..., ts-travel-service)\n"
    "- time range: 2024-01-24 03:45:00+00:00 to 2024-01-24 04:09:00+00:00\n"
    "- top kpi_name (top entries): cpu (1700), mem (1700), socket (1700), diskio (1675), workload (550), latency-50 (525), latency-90 (525), error (200)\n\n"
    "trace.csv:\n"
    "- total rows: 9968\n"
    "- distinct cmdb_id count: 21 (sample: root, ts-admin-basic-info-service, ts-admin-...)\n"
    "- time range: 2024-01-24 03:45:00+00:00 to 2024-01-24 04:09:00+00:00\n"
    "- top trace_name (top entries): trace.from_root.row_count (275), trace.from_root.error_rate (275), trace.from_root.duration_p95 (275), trace.from_root.duration_mean (275), trace.to_ts-station-service.row_count (125), trace.to_ts-station-service.error_rate (125), trace.to_ts-station-service.duration_p95 (125), trace.to_ts-station-service.duration_mean (125)\n\n"
    "log.csv:\n"
    "- total rows: 1902\n"
    "- distinct cmdb_id count: 39 (sample: ts-admin-basic-info-service, ts-admin-travel-...)\n"
    "- time range: 2024-01-24 03:45:00+00:00 to 2024-01-24 04:09:00+00:00\n"
    "- top log_name: log.error_count (951), log.row_count (951)\n\n"
    "error_logs.csv:\n"
    "- total rows: 0 (no error log records)\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   file_rows  distinct_cmdb_count                                    cmdb_ids_sample                    min_ts                    max_ts
0       8575                   68  [ts-admin-basic-info-service, ts-admin-order-s... 2024-01-24 03:45:00+00:00 2024-01-24 04:09:00+00:00,      kpi_name  count
0         cpu   1700
1         mem   1700
2      socket   1700
3      diskio   1675
4    workload    550
5  latency-50    525
6  latency-90    525
7       error    200,    file_rows  distinct_cmdb_count                                    cmdb_ids_sample                    min_ts                    max_ts
0       9968                   21  [root, ts-admin-basic-info-service, ts-admin-t... 2024-01-24 03:45:00+00:00 2024-01-24 04:09:00+00:00,                                      trace_name  count
0                     trace.from_root.row_count    275
1                    trace.from_root.error_rate    275
2                  trace.from_root.duration_p95    275
3                 trace.from_root.duration_mean    275
4         trace.to_ts-station-service.row_count    125
5        trace.to_ts-station-service.error_rate    125
6      trace.to_ts-station-service.duration_p95    125
7     trace.to_ts-station-service.duration_mean    125
8          trace.from_ts-food-service.row_count    100
9         trace.from_ts-food-service.error_rate    100
10      trace.from_ts-food-service.duration_p95    100
11     trace.from_ts-food-service.duration_mean    100
12     trace.to_ts-travel-service.duration_mean    100
13      trace.to_ts-travel-service.duration_p95    100
14        trace.to_ts-travel-service.error_rate    100
15         trace.to_ts-travel-service.row_count    100
16  trace.from_ts-travel2-service.duration_mean    100
17   trace.from_ts-travel2-service.duration_p95    100
18     trace.from_ts-travel2-service.error_rate    100
19      trace.from_ts-travel2-service.row_count    100,    file_rows  distinct_cmdb_count                                    cmdb_ids_sample                    min_ts                    max_ts
0       1902                   39  [ts-admin-basic-info-service, ts-admin-travel-... 2024-01-24 03:45:00+00:00 2024-01-24 04:09:00+00:00,           log_name  count
0  log.error_count    951
1    log.row_count    951,    file_rows  distinct_cmdb_count cmdb_ids_sample min_ts max_ts
0          0                    0              []    NaT    NaT, Empty DataFrame
Columns: [message, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing metric_df from kernel; ensure timestamp parsed
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# 1) select only the specified cmdb_ids
cmdb_list = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
selected_df = metric_df[metric_df['cmdb_id'].isin(cmdb_list)].copy()

# Determine distinct (cmdb_id, kpi_name) pairs from that selection
selected_pairs = selected_df[['cmdb_id','kpi_name']].drop_duplicates()

# 2) Compute global P95 and P5 and total_points for each pair using the full metric_df series (not time-filtered)
# Compute for all groups then filter to selected pairs to obey rule 9
all_group_stats = (
    metric_df
    .groupby(['cmdb_id','kpi_name'])['value']
    .agg(total_points='size',
         global_p95=lambda s: s.quantile(0.95),
         global_p5=lambda s: s.quantile(0.05))
    .reset_index()
)

# Keep only pairs that are in selected_pairs
pairs_key = selected_pairs.merge(all_group_stats, on=['cmdb_id','kpi_name'], how='left')
global_thresholds = pairs_key[['cmdb_id','kpi_name','global_p95','global_p5','total_points']].copy()
global_thresholds = global_thresholds.head(100)  # limit to at most 100 rows

# 3) Filter selected rows to incident window and compute aggregates per pair
start_ts = pd.Timestamp('2024-01-24 03:42:53', tz='UTC')
end_ts   = pd.Timestamp('2024-01-24 04:12:53', tz='UTC')

window_selected = selected_df[(selected_df['ts'] >= start_ts) & (selected_df['ts'] <= end_ts)].copy()

# Merge global thresholds into window rows so we can compute breach counts
window_with_thresh = window_selected.merge(
    all_group_stats,
    on=['cmdb_id','kpi_name'],
    how='left'
).rename(columns={'total_points':'_total_points_fullseries'})

# Create breach flag columns for comparisons against global thresholds
window_with_thresh['ge_p95_flag'] = (window_with_thresh['value'] >= window_with_thresh['global_p95']).astype(int)
window_with_thresh['le_p5_flag']  = (window_with_thresh['value'] <= window_with_thresh['global_p5']).astype(int)

# Group to produce compact aggregates per (cmdb_id, kpi_name)
window_aggregates = (
    window_with_thresh
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        number_of_points_in_window=('value','size'),
        earliest_timestamp_in_window=('ts','min'),
        latest_timestamp_in_window=('ts','max'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        count_ge_p95=('ge_p95_flag','sum'),
        count_le_p5=('le_p5_flag','sum')
    )
)

# Limit displayed rows to 100 for compactness
window_aggregates = window_aggregates.head(100)

# 4) Identify breach rows in the incident window: those with value >= global_p95 or <= global_p5
breaches = window_with_thresh[
    (window_with_thresh['ge_p95_flag']==1) | (window_with_thresh['le_p5_flag']==1)
].copy()

# Determine breach type and severity (absolute deviation from threshold)
def compute_breach_type_and_deviation(row):
    if row['value'] >= row['global_p95']:
        return pd.Series({'breach_type': '>=p95', 'deviation': row['value'] - row['global_p95']})
    elif row['value'] <= row['global_p5']:
        return pd.Series({'breach_type': '<=p5', 'deviation': row['global_p5'] - row['value']})
    else:
        return pd.Series({'breach_type': None, 'deviation': 0.0})

breaches[['breach_type','deviation']] = breaches.apply(compute_breach_type_and_deviation, axis=1)

# Assign priority so that '>=p95' rows come first, then '<=p5'
priority_map = {'>=p95': 0, '<=p5': 1}
breaches['priority'] = breaches['breach_type'].map(priority_map).fillna(2)

# Sort by priority then by deviation descending, take top 20
breach_rows_top20 = (
    breaches
    .sort_values(['priority','deviation'], ascending=[True, False])
    .loc[:, ['ts','cmdb_id','kpi_name','value','global_p95','global_p5','breach_type','deviation']]
    .head(20)
    .reset_index(drop=True)
)

# Final compact outputs (variables to display)
global_thresholds, window_aggregates, breach_rows_top20

```
Out[2]:
```
Summary of results (plain English):

- Scope: I analyzed metric data for 5 services: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service. There are 36 distinct (cmdb_id, kpi_name) pairs considered.

- Global thresholds (calculated using the full series for each pair): for each pair total_points = 25 (i.e., 25 samples per pair in the dataset). Global P95 and P5 were computed per (cmdb_id, kpi_name).

- Incident-window coverage: the incident window 2024-01-24 03:42:53 → 2024-01-24 04:12:53 UTC contains 25 points for each (cmdb_id, kpi_name) pair (earliest ts 2024-01-24 03:45:00+00:00, latest ts 2024-01-24 04:09:00+00:00 in the filtered data). For most pairs the aggregate counts in-window are:
  - number_of_points_in_window = 25
  - count_ge_p95 = 2
  - count_le_p5 = 2

- Breaches (top up to 20 rows sorted by severity): most severe breaches are high-memory (mem) spikes and several diskio spikes. Top breaches (service, kpi, timestamp, deviation above global P95) include:
  1. ts-auth-service, mem, 2024-01-24 03:46:00 UTC — value 240,967,300; deviation above P95 ≈ 1,544,316
  2. ts-train-service, mem, 2024-01-24 04:05:00 UTC — deviation ≈ 501,843
  3. ts-order-service, mem, 2024-01-24 04:09:00 UTC — deviation ≈ 333,797
  4. ts-order-service, mem, 2024-01-24 04:08:00 UTC — deviation ≈ 162,311
  5. ts-travel-service, mem, 2024-01-24 04:09:00 UTC — deviation ≈ 124,641
  6. ts-auth-service, mem, 2024-01-24 04:09:00 UTC — deviation ≈ 51,818
  7. ts-travel-service, mem, 2024-01-24 04:08:00 UTC — deviation ≈ 25,654
  8. ts-order-service, diskio, 2024-01-24 03:45:00 UTC — deviation ≈ 22,049
  9. ts-order-service, diskio, 2024-01-24 03:46:00 UTC — deviation ≈ 18,853
 10. ts-auth-service, diskio, 2024-01-24 03:45:00 UTC — deviation ≈ 15,684
 11. ts-travel-service, diskio, 2024-01-24 03:45:00 UTC — deviation ≈ 11,120
 12. (additional mem/diskio breaches follow in the top-20 list)

- Overall observation: the incident window contains multiple breaches above global P95 concentrated mainly in mem (memory) and diskio (disk I/O) KPIs across several services (notably ts-auth-service, ts-order-service, ts-train-service, ts-travel-service). The largest absolute deviations are memory spikes (especially ts-auth-service). A smaller set of strong disk I/O spikes also occurred (notably ts-order-service).

If you want, I can (a) show the exact compact table of all 36 (cmdb_id, kpi_name) global_p95/global_p5 values, (b) list all per-pair window aggregates, or (c) expand the full top-20 breach rows with exact numeric values. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name    global_p95     global_p5  total_points
0     ts-auth-service         cpu  1.612272e+01  1.422877e+01            25
1     ts-auth-service      diskio  5.966095e+04  4.683762e+04            25
2     ts-auth-service  latency-50  2.473207e-01  2.095155e-01            25
3     ts-auth-service  latency-90  8.999901e-01  4.668847e-01            25
4     ts-auth-service         mem  2.394230e+08  2.383904e+08            25
..                ...         ...           ...           ...           ...
31  ts-travel-service  latency-50  4.742420e-02  3.081315e-02            25
32  ts-travel-service  latency-90  9.264547e-02  8.522809e-02            25
33  ts-travel-service         mem  2.604267e+08  2.586952e+08            25
34  ts-travel-service      socket  1.375333e+01  1.171000e+01            25
35  ts-travel-service    workload  1.879383e+00  1.179253e+00            25

[36 rows x 5 columns],               cmdb_id    kpi_name  number_of_points_in_window earliest_timestamp_in_window latest_timestamp_in_window  max_value_in_window  min_value_in_window  count_ge_p95  count_le_p5
0     ts-auth-service         cpu                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00         1.818233e+01         1.401483e+01             2            2
1     ts-auth-service      diskio                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00         7.534465e+04         4.519705e+04             2            2
2     ts-auth-service  latency-50                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00         3.079208e-01         1.975016e-01             2            2
3     ts-auth-service  latency-90                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00         1.039818e+00         3.932913e-01             2            2
4     ts-auth-service         mem                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00         2.409673e+08         2.368015e+08             2            2
..                ...         ...                         ...                          ...                        ...                  ...                  ...           ...          ...
31  ts-travel-service  latency-50                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00         5.096154e-02         2.846164e-02             2            2
32  ts-travel-service  latency-90                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00         1.107143e-01         8.477935e-02             2            2
33  ts-travel-service         mem                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00         2.605513e+08         2.586092e+08             2            2
34  ts-travel-service      socket                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00         1.400000e+01         1.118333e+01             2            2
35  ts-travel-service    workload                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00         2.169983e+00         1.067000e+00             2            2

[36 rows x 9 columns],                           ts            cmdb_id kpi_name         value    global_p95     global_p5 breach_type     deviation
0  2024-01-24 03:46:00+00:00    ts-auth-service      mem  2.409673e+08  2.394230e+08  2.383904e+08       >=p95  1.544316e+06
1  2024-01-24 04:05:00+00:00   ts-train-service      mem  2.481345e+08  2.476326e+08  2.464095e+08       >=p95  5.018434e+05
2  2024-01-24 04:09:00+00:00   ts-order-service      mem  2.672981e+08  2.669643e+08  2.625173e+08       >=p95  3.337967e+05
3  2024-01-24 04:08:00+00:00   ts-order-service      mem  2.671266e+08  2.669643e+08  2.625173e+08       >=p95  1.623108e+05
4  2024-01-24 04:09:00+00:00  ts-travel-service      mem  2.605513e+08  2.604267e+08  2.586952e+08       >=p95  1.246413e+05
5  2024-01-24 04:09:00+00:00    ts-auth-service      mem  2.394748e+08  2.394230e+08  2.383904e+08       >=p95  5.181743e+04
6  2024-01-24 04:08:00+00:00  ts-travel-service      mem  2.604524e+08  2.604267e+08  2.586952e+08       >=p95  2.565461e+04
7  2024-01-24 03:45:00+00:00   ts-order-service   diskio  1.594338e+05  1.373847e+05  4.180111e+04       >=p95  2.204902e+04
8  2024-01-24 03:46:00+00:00   ts-order-service   diskio  1.562384e+05  1.373847e+05  4.180111e+04       >=p95  1.885363e+04
9  2024-01-24 03:45:00+00:00    ts-auth-service   diskio  7.534465e+04  5.966095e+04  4.683762e+04       >=p95  1.568370e+04
10 2024-01-24 03:45:00+00:00  ts-travel-service   diskio  7.127093e+04  6.015106e+04  4.205416e+04       >=p95  1.111987e+04
11 2024-01-24 04:09:00+00:00   ts-train-service      mem  2.476416e+08  2.476326e+08  2.464095e+08       >=p95  8.973274e+03
12 2024-01-24 04:04:00+00:00   ts-route-service      mem  2.595275e+08  2.595236e+08  2.586967e+08       >=p95  3.904853e+03
13 2024-01-24 04:09:00+00:00   ts-route-service      mem  2.595267e+08  2.595236e+08  2.586967e+08       >=p95  3.017387e+03
14 2024-01-24 03:46:00+00:00   ts-route-service   diskio  6.182312e+04  5.927832e+04  4.174283e+04       >=p95  2.544798e+03
15 2024-01-24 03:48:00+00:00   ts-train-service   diskio  6.328938e+04  6.233702e+04  3.598979e+04       >=p95  9.523659e+02
16 2024-01-24 03:57:00+00:00    ts-auth-service   diskio  6.059873e+04  5.966095e+04  4.683762e+04       >=p95  9.377831e+02
17 2024-01-24 04:04:00+00:00   ts-route-service   diskio  5.969003e+04  5.927832e+04  4.174283e+04       >=p95  4.117032e+02
18 2024-01-24 03:45:00+00:00   ts-train-service   diskio  6.252507e+04  6.233702e+04  3.598979e+04       >=p95  1.880507e+02
19 2024-01-24 04:09:00+00:00  ts-travel-service   diskio  6.016625e+04  6.015106e+04  4.205416e+04       >=p95  1.519301e+01)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# Ensure trace_df is loaded and timestamp column parsed to UTC datetimes
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# 1) Select trace rows where trace_name contains any of the given substrings
substrs = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']
# build regex to match any substring
pattern = '|'.join([pd.util.testing.rands(0) for _ in range(0)])  # placeholder to ensure usage below (will be overwritten)
pattern = '(' + '|'.join(substrs) + ')'
selected_traces = trace_df[trace_df['trace_name'].str.contains(pattern, na=False)].copy()

# Distinct trace_name values found and their counts (limit to top 50)
distinct_trace_counts = selected_traces['trace_name'].value_counts().head(50).reset_index()
distinct_trace_counts.columns = ['trace_name', 'count']

# 2) For each distinct trace_name from step 1, compute global P95 and total_points using full series
selected_trace_names = distinct_trace_counts['trace_name'].unique().tolist()
# Use full trace_df series for calculation of thresholds (rule 9)
global_thresholds_trace = (
    trace_df[trace_df['trace_name'].isin(selected_trace_names)]
    .groupby('trace_name', as_index=False)['value']
    .agg(global_p95=lambda s: s.quantile(0.95), total_points='size')
    .loc[:, ['trace_name','global_p95','total_points']]
    .head(100)
)

# 3) Filter the selected trace rows to the incident window and compute aggregates
start_ts = pd.Timestamp('2024-01-24 03:42:53', tz='UTC')
end_ts   = pd.Timestamp('2024-01-24 04:12:53', tz='UTC')

window_selected_traces = selected_traces[(selected_traces['ts'] >= start_ts) & (selected_traces['ts'] <= end_ts)].copy()

# Merge global_p95 into window rows
window_with_thresh = window_selected_traces.merge(global_thresholds_trace, on='trace_name', how='left')

# Flag breaches >= global_p95
window_with_thresh['ge_p95_flag'] = (window_with_thresh['value'] >= window_with_thresh['global_p95']).astype(int)

# Aggregates per trace_name
window_aggregates_trace = (
    window_with_thresh
    .groupby('trace_name', as_index=False)
    .agg(
        number_of_points_in_window=('value','size'),
        earliest_timestamp_in_window=('ts','min'),
        latest_timestamp_in_window=('ts','max'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        count_ge_p95=('ge_p95_flag','sum')
    )
    .head(100)
)

# 4) Top 20 incident-window trace rows that breach >=global_p95, sorted by severity:
# severity: first by count_ge_p95 (per-trace_name) then by deviation (value - global_p95)
# Merge count_ge_p95 into individual breach rows
breaches = window_with_thresh[window_with_thresh['ge_p95_flag'] == 1].copy()
# compute deviation
breaches['deviation'] = breaches['value'] - breaches['global_p95']

# bring in count_ge_p95 per trace_name for severity sorting
trace_breach_counts = window_aggregates_trace.loc[:, ['trace_name','count_ge_p95']]
breaches = breaches.merge(trace_breach_counts, on='trace_name', how='left')

# Sort by count_ge_p95 descending then deviation descending, limit to top 20
breach_rows_top20_trace = (
    breaches
    .sort_values(['count_ge_p95','deviation'], ascending=[False, False])
    .loc[:, ['ts','cmdb_id','trace_name','value','global_p95','deviation']]
    .head(20)
    .reset_index(drop=True)
)

# Display compact aggregates only
distinct_trace_counts, global_thresholds_trace, window_aggregates_trace, breach_rows_top20_trace

```
Out[3]:
```
summary = (
    "Trace analysis summary (UTC):\n\n"
    "- Selection: 40 distinct trace_name values matched the five services (examples with highest counts: "
    "trace.from_ts-travel-service.{duration_mean,duration_p95,error_rate,row_count} each 100 records).\n\n"
    "- Global thresholds: P95 computed per trace_name (examples: "
    "trace.from_ts-auth-service.duration_mean global_p95=0.003890 (25 pts); "
    "trace.from_ts-order-service.duration_mean global_p95=0.218602 (50 pts); "
    "trace.to_ts-travel-service.duration_mean global_p95=0.080131 (100 pts)).\n\n"
    "- Incident-window aggregates (2024-01-24 03:42:53 → 04:12:53 UTC): each trace_name had full coverage in-window (typical counts: 25, 50, 75, or 100 points). "
    "Many traces show a small number of points >= global P95 (e.g., duration_mean and row_count traces with count_ge_p95 between 2–5). "
    "trace.*.error_rate series are 0.0 in the window (global_p95 == 0), so they trivially appear as >=P95 but do not indicate real errors.\n\n"
    "- Top breaches (incident window, sorted by severity): the highest-value deviations are mostly in duration_mean and row_count metrics and not in error_rate. "
    "Notable items: multiple duration_mean breaches for travel/order services (e.g., trace.from_ts-order-service.duration_mean had max ~0.232s and count_ge_p95=3), "
    "trace.to_ts-travel-service.duration_mean had count_ge_p95=5, and several row_count traces (to_ts-train/to_ts-travel) show high row counts (up to ~440) with count_ge_p95 up to 5.\n\n"
    "Conclusion: there are no obvious trace error spikes (error_rate is zero). The actionable signals are increased latencies (duration_mean/p95) and elevated row_count for travel/train related traces. "
    "Recommend investigating travel/order/train service interactions and the elevated request volumes and latencies for those traces."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                                    trace_name  count
0   trace.from_ts-travel-service.duration_mean    100
1    trace.from_ts-travel-service.duration_p95    100
2      trace.from_ts-travel-service.error_rate    100
3       trace.from_ts-travel-service.row_count    100
4         trace.to_ts-travel-service.row_count    100
..                                         ...    ...
35   trace.from_ts-route-service.duration_mean     25
36       trace.from_ts-train-service.row_count     25
37      trace.from_ts-train-service.error_rate     25
38    trace.from_ts-train-service.duration_p95     25
39   trace.from_ts-train-service.duration_mean     25

[40 rows x 2 columns],                                    trace_name  global_p95  total_points
0    trace.from_ts-auth-service.duration_mean    0.003890            25
1     trace.from_ts-auth-service.duration_p95    0.002097            25
2       trace.from_ts-auth-service.error_rate    0.000000            25
3        trace.from_ts-auth-service.row_count  369.600000            25
4   trace.from_ts-order-service.duration_mean    0.218602            50
..                                        ...         ...           ...
35        trace.to_ts-train-service.row_count  440.000000            75
36   trace.to_ts-travel-service.duration_mean    0.080131           100
37    trace.to_ts-travel-service.duration_p95    0.095575           100
38      trace.to_ts-travel-service.error_rate    0.000000           100
39       trace.to_ts-travel-service.row_count  339.200000           100

[40 rows x 3 columns],                                    trace_name  number_of_points_in_window earliest_timestamp_in_window latest_timestamp_in_window  max_value_in_window  min_value_in_window  count_ge_p95
0    trace.from_ts-auth-service.duration_mean                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00             0.004143             0.001558             2
1     trace.from_ts-auth-service.duration_p95                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00             0.002141             0.001854             2
2       trace.from_ts-auth-service.error_rate                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00             0.000000             0.000000            25
3        trace.from_ts-auth-service.row_count                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00           370.000000            45.000000             2
4   trace.from_ts-order-service.duration_mean                          50    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00             0.231988             0.004116             3
..                                        ...                         ...                          ...                        ...                  ...                  ...           ...
35        trace.to_ts-train-service.row_count                          75    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00           440.000000             5.000000             5
36   trace.to_ts-travel-service.duration_mean                         100    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00             0.097668             0.005214             5
37    trace.to_ts-travel-service.duration_p95                         100    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00             0.183330             0.011715             5
38      trace.to_ts-travel-service.error_rate                         100    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00             0.000000             0.000000           100
39       trace.to_ts-travel-service.row_count                         100    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00           432.000000             1.000000             5

[40 rows x 7 columns],                           ts                  cmdb_id                               trace_name  value  global_p95  deviation
0  2024-01-24 03:45:00+00:00                     root    trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
1  2024-01-24 03:45:00+00:00  ts-admin-travel-service    trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
2  2024-01-24 03:45:00+00:00          ts-food-service    trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
3  2024-01-24 03:45:00+00:00         ts-route-service  trace.from_ts-travel-service.error_rate    0.0         0.0        0.0
4  2024-01-24 03:45:00+00:00    ts-ticketinfo-service  trace.from_ts-travel-service.error_rate    0.0         0.0        0.0
5  2024-01-24 03:45:00+00:00         ts-train-service  trace.from_ts-travel-service.error_rate    0.0         0.0        0.0
6  2024-01-24 03:45:00+00:00        ts-travel-service  trace.from_ts-travel-service.error_rate    0.0         0.0        0.0
7  2024-01-24 03:45:00+00:00        ts-travel-service    trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
8  2024-01-24 03:46:00+00:00                     root    trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
9  2024-01-24 03:46:00+00:00  ts-admin-travel-service    trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
10 2024-01-24 03:46:00+00:00          ts-food-service    trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
11 2024-01-24 03:46:00+00:00         ts-route-service  trace.from_ts-travel-service.error_rate    0.0         0.0        0.0
12 2024-01-24 03:46:00+00:00    ts-ticketinfo-service  trace.from_ts-travel-service.error_rate    0.0         0.0        0.0
13 2024-01-24 03:46:00+00:00         ts-train-service  trace.from_ts-travel-service.error_rate    0.0         0.0        0.0
14 2024-01-24 03:46:00+00:00        ts-travel-service  trace.from_ts-travel-service.error_rate    0.0         0.0        0.0
15 2024-01-24 03:46:00+00:00        ts-travel-service    trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
16 2024-01-24 03:47:00+00:00                     root    trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
17 2024-01-24 03:47:00+00:00  ts-admin-travel-service    trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
18 2024-01-24 03:47:00+00:00          ts-food-service    trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
19 2024-01-24 03:47:00+00:00         ts-route-service  trace.from_ts-travel-service.error_rate    0.0         0.0        0.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse kernel variables; ensure metric_df and timestamp column exist
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure global thresholds (all_group_stats) exist; if not, compute using entire metric_df (rule 9)
if 'all_group_stats' not in globals():
    all_group_stats = (
        metric_df
        .groupby(['cmdb_id','kpi_name'])['value']
        .agg(total_points='size',
             global_p95=lambda s: s.quantile(0.95),
             global_p5=lambda s: s.quantile(0.05))
        .reset_index()
    )

# 1) Restrict to given services and KPIs, and to the incident window (UTC)
cmdb_list = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
kpi_list = ["mem","diskio"]
start_ts = pd.Timestamp('2024-01-24 03:42:53', tz='UTC')
end_ts   = pd.Timestamp('2024-01-24 04:12:53', tz='UTC')

df_sel = metric_df[
    metric_df['cmdb_id'].isin(cmdb_list) &
    metric_df['kpi_name'].isin(kpi_list)
].copy()

# Merge global_p95 from all_group_stats (calculated over full series) before time filtering
# all_group_stats might have several columns; ensure global_p95 present
if 'global_p95' not in all_group_stats.columns:
    # recompute minimal thresholds table if needed
    all_group_stats = (
        metric_df
        .groupby(['cmdb_id','kpi_name'])['value']
        .agg(global_p95=lambda s: s.quantile(0.95), total_points='size')
        .reset_index()
    )

df_sel = df_sel.merge(all_group_stats[['cmdb_id','kpi_name','global_p95']], on=['cmdb_id','kpi_name'], how='left')

# Now filter to incident window (rule 10)
df_window = df_sel[(df_sel['ts'] >= start_ts) & (df_sel['ts'] <= end_ts)].copy()

# 2) For each (cmdb_id,kpi_name) identify consecutive runs where value >= global_p95
df_window['is_breach'] = df_window['value'] >= df_window['global_p95']

# We'll collect runs in a DataFrame
runs = []

# Process group by group to detect consecutive one-minute adjacency
for (cmdb, kpi), grp in df_window.groupby(['cmdb_id','kpi_name'], sort=False):
    g = grp.sort_values('ts').reset_index(drop=True)
    # consider only breach rows
    breaches = g[g['is_breach']].copy()
    if breaches.empty:
        continue
    # compute time diffs
    breaches['prev_ts'] = breaches['ts'].shift(1)
    breaches['diff_min'] = (breaches['ts'] - breaches['prev_ts']) / pd.Timedelta(minutes=1)
    # start a new run where diff != 1 (or NaN for first)
    breaches['new_run'] = (breaches['diff_min'].isna()) | (breaches['diff_min'] != 1)
    breaches['run_id'] = breaches['new_run'].cumsum()
    # aggregate per run
    agg = (
        breaches
        .groupby('run_id', as_index=False)
        .agg(
            cmdb_id=('cmdb_id','first'),
            kpi_name=('kpi_name','first'),
            start_timestamp=('ts','min'),
            end_timestamp=('ts','max'),
            length_in_points=('ts','size'),
            max_value_in_run=('value','max'),
            global_p95=('global_p95','first')
        )
    )
    # compute relative_deviation, guard divide-by-zero
    agg['relative_deviation'] = np.where(
        agg['global_p95'] != 0,
        (agg['max_value_in_run'] - agg['global_p95']) / agg['global_p95'],
        np.inf
    )
    runs.append(agg)

# Combine runs
if runs:
    runs_df = pd.concat(runs, ignore_index=True)
else:
    runs_df = pd.DataFrame(columns=['cmdb_id','kpi_name','start_timestamp','end_timestamp',
                                    'length_in_points','max_value_in_run','global_p95','relative_deviation'])

# 3) Filter out runs of length 1 if relative_deviation <= 0.5
cond_keep = (runs_df['length_in_points'] > 1) | (runs_df['relative_deviation'] > 0.5)
runs_filtered = runs_df[cond_keep].copy()

# 4) Return compact table limited to top 20 rows sorted by relative_deviation desc
runs_top20 = runs_filtered.sort_values('relative_deviation', ascending=False).head(20).reset_index(drop=True)

# Short summary: earliest fault start time, and the run with largest relative_deviation
if not runs_filtered.empty:
    earliest_start = runs_filtered['start_timestamp'].min()
    top_run = runs_filtered.loc[runs_filtered['relative_deviation'].idxmax()]
    summary = pd.DataFrame([{
        'earliest_fault_start_time': earliest_start,
        'top_run_cmdb_id': top_run['cmdb_id'],
        'top_run_kpi_name': top_run['kpi_name'],
        'top_run_start_timestamp': top_run['start_timestamp'],
        'top_run_relative_deviation': top_run['relative_deviation']
    }])
else:
    summary = pd.DataFrame([{
        'earliest_fault_start_time': pd.NaT,
        'top_run_cmdb_id': None,
        'top_run_kpi_name': None,
        'top_run_start_timestamp': pd.NaT,
        'top_run_relative_deviation': np.nan
    }])

# Display compact outputs (DataFrames)
runs_top20, summary

```
Out[4]:
```
text_summary = (
    "Fault-run summary (mem and diskio for the five services, incident window 2024-01-24 03:42:53→04:12:53 UTC):\n\n"
    "- Total fault runs detected after filtering: 3 (all runs have length 2 points).\n\n"
    "- Runs (cmdb_id, kpi, start → end, length, max_value, global_p95, relative_deviation):\n"
    "  1) ts-order-service, diskio: 2024-01-24 03:45 → 2024-01-24 03:46, length=2, max=159433.8, p95=137384.7, rel_dev≈0.1605 (≈16.05%)\n"
    "  2) ts-order-service, mem:   2024-01-24 04:08 → 2024-01-24 04:09, length=2, max=267298100, p95=266964300, rel_dev≈0.00125 (≈0.125%)\n"
    "  3) ts-travel-service, mem:  2024-01-24 04:08 → 2024-01-24 04:09, length=2, max=260551300, p95=260426700, rel_dev≈0.00048 (≈0.048%)\n\n"
    "- Earliest fault start time: 2024-01-24 03:45:00 UTC.\n"
    "- Most severe run (largest relative_deviation): ts-order-service, diskio starting 2024-01-24 03:45:00 UTC (rel_dev ≈ 0.1605, ~16%).\n\n"
    "Interpretation: The clearest multi-point fault is a disk I/O spike on ts-order-service (~16% above its P95 for two consecutive minutes). Memory breaches are present but their deviations above P95 are very small (<<1%), likely low-impact isolated increases."
)
text_summary

The original code execution output of IPython Kernel is also provided below for reference:

(   run_id            cmdb_id kpi_name           start_timestamp             end_timestamp  length_in_points  max_value_in_run    global_p95  relative_deviation
0       1   ts-order-service   diskio 2024-01-24 03:45:00+00:00 2024-01-24 03:46:00+00:00                 2      1.594338e+05  1.373847e+05            0.160491
1       1   ts-order-service      mem 2024-01-24 04:08:00+00:00 2024-01-24 04:09:00+00:00                 2      2.672981e+08  2.669643e+08            0.001250
2       1  ts-travel-service      mem 2024-01-24 04:08:00+00:00 2024-01-24 04:09:00+00:00                 2      2.605513e+08  2.604267e+08            0.000479,   earliest_fault_start_time   top_run_cmdb_id top_run_kpi_name   top_run_start_timestamp  top_run_relative_deviation
0 2024-01-24 03:45:00+00:00  ts-order-service           diskio 2024-01-24 03:45:00+00:00                    0.160491)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Ensure log_df exists and timestamps parsed
if 'log_df' not in globals():
    log_df = pd.read_csv("log.csv")
if 'ts' not in log_df.columns:
    log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# 1) Select rows for the five services
cmdb_list = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
selected_logs = log_df[log_df['cmdb_id'].isin(cmdb_list)].copy()

# 2) Compute global_p95 and global_p5 using the full available series (for each (cmdb_id, log_name))
global_log_thresh = (
    log_df
    .groupby(['cmdb_id','log_name'], as_index=False)['value']
    .agg(global_p95=lambda s: s.quantile(0.95),
         global_p5 = lambda s: s.quantile(0.05),
         total_points='size')
    .loc[:, ['cmdb_id','log_name','global_p95','global_p5','total_points']]
)

# Keep only thresholds for pairs present in our selected set (compact)
pairs_in_selection = selected_logs[['cmdb_id','log_name']].drop_duplicates()
global_thresh_selected = pairs_in_selection.merge(global_log_thresh, on=['cmdb_id','log_name'], how='left').head(100)

# 3) Filter selected rows to the incident window and compute aggregates
start_ts = pd.Timestamp('2024-01-24 03:42:53', tz='UTC')
end_ts   = pd.Timestamp('2024-01-24 04:12:53', tz='UTC')

# Merge thresholds into selected logs BEFORE time filtering (rule 9)
selected_with_thresh = selected_logs.merge(global_log_thresh, on=['cmdb_id','log_name'], how='left')

window_logs = selected_with_thresh[(selected_with_thresh['ts'] >= start_ts) & (selected_with_thresh['ts'] <= end_ts)].copy()

# Flag breaches
window_logs['ge_p95'] = (window_logs['value'] >= window_logs['global_p95']).astype(int)
window_logs['le_p5']  = (window_logs['value'] <= window_logs['global_p5']).astype(int)

# Aggregates per (cmdb_id, log_name)
window_aggregates = (
    window_logs
    .groupby(['cmdb_id','log_name'], as_index=False)
    .agg(
        number_of_points_in_window=('value','size'),
        earliest_timestamp_in_window=('ts','min'),
        latest_timestamp_in_window=('ts','max'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        count_ge_p95=('ge_p95','sum'),
        count_le_p5=('le_p5','sum')
    )
    .head(100)
)

# 4) Top up to 20 incident-window log rows that breach >=global_p95 or <=global_p5
breaches = window_logs[(window_logs['ge_p95']==1) | (window_logs['le_p5']==1)].copy()

if not breaches.empty:
    def determine_breach_and_deviation(row):
        if row['ge_p95'] == 1:
            return pd.Series({'breach_type': '>=p95', 'deviation': row['value'] - row['global_p95']})
        elif row['le_p5'] == 1:
            return pd.Series({'breach_type': '<=p5', 'deviation': row['value'] - row['global_p5']})
        else:
            return pd.Series({'breach_type': None, 'deviation': 0.0})

    breaches[['breach_type','deviation']] = breaches.apply(determine_breach_and_deviation, axis=1)
    # Priority: '>=p95' first, then '<=p5'; then by absolute deviation descending
    priority_map = {'>=p95': 0, '<=p5': 1}
    breaches['priority'] = breaches['breach_type'].map(priority_map).fillna(2)
    breaches['abs_deviation'] = breaches['deviation'].abs()
    breach_rows_top20 = (
        breaches
        .sort_values(['priority', 'abs_deviation'], ascending=[True, False])
        .loc[:, ['ts','cmdb_id','log_name','value','global_p95','global_p5','breach_type','deviation']]
        .head(20)
        .reset_index(drop=True)
    )
else:
    breach_rows_top20 = breaches.loc[:, ['ts','cmdb_id','log_name','value','global_p95','global_p5']].copy()
    # add expected columns if empty
    for col in ['breach_type','deviation']:
        if col not in breach_rows_top20.columns:
            breach_rows_top20[col] = pd.Series(dtype='object')

# Compact outputs (variables to display)
global_thresh_selected, window_aggregates, breach_rows_top20

```
Out[5]:
```
Summary of log analysis (UTC):

- Scope: analyzed logs for ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service within 2024-01-24 03:42:53 → 04:12:53 UTC.

- Global thresholds (full series): each (cmdb_id, log_name) has 25 points. For log.row_count P95 values:
  - ts-travel-service row_count P95 = 585.2
  - ts-route-service row_count P95 = 375.8
  - ts-auth-service row_count P95 = 369.6
  - ts-order-service row_count P95 = 62.4
  - (log.error_count P95 = 0.0 for all services)

- Incident-window aggregates (per pair): each pair has 25 points in-window (earliest 2024-01-24 03:45:00, latest 2024-01-24 04:09:00). For row_count, in-window max values:
  - ts-travel-service: max 650
  - ts-route-service: max 382
  - ts-auth-service: max 370
  - ts-order-service: max 67

- Breaches (value >= global_p95) — top items:
  1. ts-travel-service, log.row_count, 2024-01-24 03:52:00 — value 650 (deviation +64.8 above P95)
  2. ts-travel-service, log.row_count, 2024-01-24 03:53:00 — value 593 (deviation +7.8)
  3. ts-route-service, log.row_count, 2024-01-24 03:52:00 — value 382 (deviation +6.2)
  4. ts-order-service, log.row_count, 2024-01-24 03:55:00 — value 67 (deviation +4.6)
  5. ts-order-service, log.row_count, 2024-01-24 03:56:00 — value 63 (deviation +0.6)
  6. ts-auth-service, log.row_count, 2024-01-24 03:53:00 and 04:05:00 — value 370 (deviation +0.4)
  (multiple log.error_count rows show 0.0 and are flagged because global_p95=0 — these are not informative.)

- Interpretation / actionable insight:
  - The strongest, non-trivial signal is elevated log.row_count for ts-travel-service around 03:52–03:53 (peak 650, ~11% above its P95). ts-route-service and ts-order-service also show smaller row_count spikes.
  - log.error_count is zero across these services (P95=0), so error_count does not indicate application errors during the window.
  - Recommend investigating increased request/processing volume for ts-travel-service (and related ts-route/ts-order interactions) around 03:52 UTC, which may explain downstream latency or resource symptoms seen elsewhere.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  global_p95  global_p5  total_points
0    ts-auth-service  log.error_count         0.0        0.0            25
1    ts-auth-service    log.row_count       369.6      317.2            25
2   ts-order-service  log.error_count         0.0        0.0            25
3   ts-order-service    log.row_count        62.4       13.0            25
4   ts-route-service  log.error_count         0.0        0.0            25
5   ts-route-service    log.row_count       375.8      276.2            25
6  ts-travel-service  log.error_count         0.0        0.0            25
7  ts-travel-service    log.row_count       585.2      337.8            25,              cmdb_id         log_name  number_of_points_in_window earliest_timestamp_in_window latest_timestamp_in_window  max_value_in_window  min_value_in_window  count_ge_p95  count_le_p5
0    ts-auth-service  log.error_count                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00                    0                    0            25           25
1    ts-auth-service    log.row_count                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00                  370                   46             2            2
2   ts-order-service  log.error_count                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00                    0                    0            25           25
3   ts-order-service    log.row_count                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00                   67                    5             2            2
4   ts-route-service  log.error_count                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00                    0                    0            25           25
5   ts-route-service    log.row_count                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00                  382                   43             2            2
6  ts-travel-service  log.error_count                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00                    0                    0            25           25
7  ts-travel-service    log.row_count                          25    2024-01-24 03:45:00+00:00  2024-01-24 04:09:00+00:00                  650                   47             2            2,                           ts            cmdb_id         log_name  value  global_p95  global_p5 breach_type  deviation
0  2024-01-24 03:52:00+00:00  ts-travel-service    log.row_count    650       585.2      337.8       >=p95       64.8
1  2024-01-24 03:53:00+00:00  ts-travel-service    log.row_count    593       585.2      337.8       >=p95        7.8
2  2024-01-24 03:52:00+00:00   ts-route-service    log.row_count    382       375.8      276.2       >=p95        6.2
3  2024-01-24 03:55:00+00:00   ts-order-service    log.row_count     67        62.4       13.0       >=p95        4.6
4  2024-01-24 03:56:00+00:00   ts-order-service    log.row_count     63        62.4       13.0       >=p95        0.6
5  2024-01-24 03:53:00+00:00    ts-auth-service    log.row_count    370       369.6      317.2       >=p95        0.4
6  2024-01-24 04:05:00+00:00    ts-auth-service    log.row_count    370       369.6      317.2       >=p95        0.4
7  2024-01-24 03:57:00+00:00   ts-route-service    log.row_count    376       375.8      276.2       >=p95        0.2
8  2024-01-24 03:45:00+00:00    ts-auth-service  log.error_count      0         0.0        0.0       >=p95        0.0
9  2024-01-24 03:45:00+00:00   ts-order-service  log.error_count      0         0.0        0.0       >=p95        0.0
10 2024-01-24 03:45:00+00:00   ts-route-service  log.error_count      0         0.0        0.0       >=p95        0.0
11 2024-01-24 03:45:00+00:00  ts-travel-service  log.error_count      0         0.0        0.0       >=p95        0.0
12 2024-01-24 03:46:00+00:00    ts-auth-service  log.error_count      0         0.0        0.0       >=p95        0.0
13 2024-01-24 03:46:00+00:00   ts-order-service  log.error_count      0         0.0        0.0       >=p95        0.0
14 2024-01-24 03:46:00+00:00   ts-route-service  log.error_count      0         0.0        0.0       >=p95        0.0
15 2024-01-24 03:46:00+00:00  ts-travel-service  log.error_count      0         0.0        0.0       >=p95        0.0
16 2024-01-24 03:47:00+00:00    ts-auth-service  log.error_count      0         0.0        0.0       >=p95        0.0
17 2024-01-24 03:47:00+00:00   ts-order-service  log.error_count      0         0.0        0.0       >=p95        0.0
18 2024-01-24 03:47:00+00:00   ts-route-service  log.error_count      0         0.0        0.0       >=p95        0.0
19 2024-01-24 03:47:00+00:00  ts-travel-service  log.error_count      0         0.0        0.0       >=p95        0.0)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# ensure trace_df and timestamp parsed
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# substrings to match
substrs = [
    'to_ts-order-service','from_ts-order-service',
    'to_ts-travel-service','from_ts-travel-service',
    'to_ts-train-service','from_ts-train-service',
    'to_ts-route-service','from_ts-route-service'
]
pattern = '|'.join(substrs)

# 1) select rows whose trace_name contains any substring
selected_traces2 = trace_df[trace_df['trace_name'].str.contains(pattern, na=False)].copy()
selected_trace_names = selected_traces2['trace_name'].unique().tolist()

# 2) compute global_p95 for each selected trace_name using the full trace_df series (rule 9)
global_thresholds_trace_sel = (
    trace_df[trace_df['trace_name'].isin(selected_trace_names)]
    .groupby('trace_name', as_index=False)['value']
    .agg(global_p95=lambda s: s.quantile(0.95), total_points='size')
)

# 3) filter to incident window and merge thresholds
start_ts = pd.Timestamp('2024-01-24 03:42:53', tz='UTC')
end_ts   = pd.Timestamp('2024-01-24 04:12:53', tz='UTC')

window_sel = selected_traces2[(selected_traces2['ts'] >= start_ts) & (selected_traces2['ts'] <= end_ts)].copy()
window_with_thresh = window_sel.merge(global_thresholds_trace_sel, on='trace_name', how='left')

# compute ge_p95 flag
window_with_thresh['ge_p95_flag'] = (window_with_thresh['value'] >= window_with_thresh['global_p95']).astype(int)

# aggregate per trace_name: total_points_in_window, count_ge_p95, earliest_breach_timestamp, latest_breach_timestamp, max_value_in_window
rows = []
for name, grp in window_with_thresh.groupby('trace_name', sort=True):
    total_pts = len(grp)
    count_ge = int(grp['ge_p95_flag'].sum())
    max_val = grp['value'].max() if total_pts>0 else np.nan
    # determine earliest/latest breach timestamps or NaT
    if count_ge > 0:
        earliest_breach = grp.loc[grp['ge_p95_flag'] == 1, 'ts'].min()
        latest_breach = grp.loc[grp['ge_p95_flag'] == 1, 'ts'].max()
    else:
        earliest_breach = pd.NaT
        latest_breach = pd.NaT
    rows.append({
        'trace_name': name,
        'total_points_in_window': total_pts,
        'count_ge_p95': count_ge,
        'earliest_breach_timestamp': earliest_breach,
        'latest_breach_timestamp': latest_breach,
        'max_value_in_window': max_val
    })

aggregates_df = pd.DataFrame(rows).sort_values('count_ge_p95', ascending=False).reset_index(drop=True).head(50)

# 4) example breach rows across these trace_names: value >= global_p95, compute deviation and sort by deviation desc
breaches = window_with_thresh[window_with_thresh['ge_p95_flag'] == 1].copy()
if not breaches.empty:
    breaches['deviation'] = breaches['value'] - breaches['global_p95']
    breach_rows_top20 = (
        breaches
        .sort_values('deviation', ascending=False)
        .loc[:, ['ts','cmdb_id','trace_name','value','global_p95','deviation']]
        .head(20)
        .reset_index(drop=True)
    )
else:
    breach_rows_top20 = breaches.loc[:, ['ts','cmdb_id','trace_name','value','global_p95']].copy()
    breach_rows_top20['deviation'] = pd.Series(dtype='float')

# Final compact outputs
aggregates_df, breach_rows_top20

```
Out[6]:
```
Summary of trace-window analysis (UTC 2024-01-24 03:42:53 → 04:12:53):

- Scope: I analyzed traces whose names reference order, travel, train, or route service interactions (both "to_" and "from_") within the incident window.

- Aggregate observation:
  - 32 distinct trace_name groups matched the selection.
  - Many trace.error_rate series are zero throughout the window (global P95 = 0), so they trivially appear as breaches but carry no error signal.
  - The strongest, meaningful signals are in trace.row_count and some duration metrics (duration_mean / duration_p95) for travel/route/order/train interactions.

- Key breach / spike examples (value, global_p95, deviation = value - global_p95):
  1. trace.from_ts-travel-service.row_count (ts-travel-service @ 03:52:00): 432.0 vs P95=339.2, deviation = +92.8
  2. trace.to_ts-travel-service.row_count   (ts-travel-service @ 03:52:00): 432.0 vs P95=339.2, deviation = +92.8
  3. trace.to_ts-travel-service.row_count   (ts-travel-service @ 03:53:00): 393.0 vs P95=339.2, deviation = +53.8
  4. trace.to_ts-route-service.row_count    (ts-route-service  @ 03:52:00): 764.0 vs P95=719.2, deviation = +44.8
  5. trace.from_ts-travel-service.row_count (ts-travel-service @ 03:46:00): 373.0 vs P95=339.2, deviation = +33.8
  - Additional row_count spikes for route/travel/order appear around 03:46–03:57 and at 03:52–03:53.

- Timing: the most pronounced volume spikes occur around 03:52–03:53 UTC (travel and route row counts) and some repeated higher-than-P95 counts through ~03:57 and later minutes.

- Conclusion / recommendation:
  - No trace error-rate spikes (error_rate = 0). The main anomaly is elevated request/row counts (and some increased durations) for travel and route traces, peaking ~03:52–03:53 UTC.
  - Investigate increased traffic or batching behavior in ts-travel-service and ts-route-service around 03:52 UTC and downstream processing for ts-order/ts-train interactions, as these volume spikes likely explain related latency/resource signals seen elsewhere.

The original code execution output of IPython Kernel is also provided below for reference:

(                                   trace_name  total_points_in_window  count_ge_p95 earliest_breach_timestamp   latest_breach_timestamp  max_value_in_window
0     trace.from_ts-travel-service.error_rate                     100           100 2024-01-24 03:45:00+00:00 2024-01-24 04:09:00+00:00             0.000000
1       trace.to_ts-travel-service.error_rate                     100           100 2024-01-24 03:45:00+00:00 2024-01-24 04:09:00+00:00             0.000000
2        trace.to_ts-route-service.error_rate                      75            75 2024-01-24 03:45:00+00:00 2024-01-24 04:09:00+00:00             0.000000
3        trace.to_ts-train-service.error_rate                      75            75 2024-01-24 03:45:00+00:00 2024-01-24 04:09:00+00:00             0.000000
4        trace.to_ts-order-service.error_rate                      74            74 2024-01-24 03:45:00+00:00 2024-01-24 04:09:00+00:00             0.000000
..                                        ...                     ...           ...                       ...                       ...                  ...
27   trace.from_ts-train-service.duration_p95                      25             2 2024-01-24 03:51:00+00:00 2024-01-24 04:00:00+00:00             0.002108
28  trace.from_ts-route-service.duration_mean                      25             2 2024-01-24 03:45:00+00:00 2024-01-24 04:00:00+00:00             0.001402
29  trace.from_ts-train-service.duration_mean                      25             2 2024-01-24 03:59:00+00:00 2024-01-24 04:00:00+00:00             0.001591
30   trace.from_ts-route-service.duration_p95                      25             2 2024-01-24 04:00:00+00:00 2024-01-24 04:01:00+00:00             0.001890
31      trace.from_ts-route-service.row_count                      25             2 2024-01-24 03:52:00+00:00 2024-01-24 03:57:00+00:00           764.000000

[32 rows x 6 columns],                           ts            cmdb_id                              trace_name  value  global_p95  deviation
0  2024-01-24 03:52:00+00:00  ts-travel-service  trace.from_ts-travel-service.row_count  432.0       339.2       92.8
1  2024-01-24 03:52:00+00:00  ts-travel-service    trace.to_ts-travel-service.row_count  432.0       339.2       92.8
2  2024-01-24 03:53:00+00:00  ts-travel-service    trace.to_ts-travel-service.row_count  393.0       339.2       53.8
3  2024-01-24 03:53:00+00:00  ts-travel-service  trace.from_ts-travel-service.row_count  393.0       339.2       53.8
4  2024-01-24 03:52:00+00:00   ts-route-service     trace.to_ts-route-service.row_count  764.0       719.2       44.8
5  2024-01-24 03:46:00+00:00  ts-travel-service  trace.from_ts-travel-service.row_count  373.0       339.2       33.8
6  2024-01-24 03:46:00+00:00  ts-travel-service    trace.to_ts-travel-service.row_count  373.0       339.2       33.8
7  2024-01-24 03:57:00+00:00   ts-route-service     trace.to_ts-route-service.row_count  752.0       719.2       32.8
8  2024-01-24 03:53:00+00:00   ts-route-service     trace.to_ts-route-service.row_count  750.0       719.2       30.8
9  2024-01-24 04:03:00+00:00  ts-travel-service    trace.to_ts-travel-service.row_count  353.0       339.2       13.8
10 2024-01-24 04:03:00+00:00  ts-travel-service  trace.from_ts-travel-service.row_count  353.0       339.2       13.8
11 2024-01-24 03:52:00+00:00   ts-route-service   trace.from_ts-route-service.row_count  764.0       751.6       12.4
12 2024-01-24 03:55:00+00:00   ts-order-service     trace.to_ts-order-service.row_count   59.0        47.0       12.0
13 2024-01-24 03:56:00+00:00   ts-order-service     trace.to_ts-order-service.row_count   56.0        47.0        9.0
14 2024-01-24 03:48:00+00:00   ts-order-service     trace.to_ts-order-service.row_count   55.0        47.0        8.0
15 2024-01-24 03:55:00+00:00   ts-order-service   trace.from_ts-order-service.row_count   59.0        51.4        7.6
16 2024-01-24 03:56:00+00:00   ts-order-service   trace.from_ts-order-service.row_count   56.0        51.4        4.6
17 2024-01-24 03:59:00+00:00  ts-travel-service    trace.to_ts-travel-service.row_count  343.0       339.2        3.8
18 2024-01-24 03:59:00+00:00  ts-travel-service  trace.from_ts-travel-service.row_count  343.0       339.2        3.8
19 2024-01-24 03:48:00+00:00   ts-order-service   trace.from_ts-order-service.row_count   55.0        51.4        3.6)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```